In [ ]:
!pip install ultralytics opencv-python-headless numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 13.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [ultralytics]


In [ ]:
import os
import cv2
import numpy as np
import urllib.request
from ultralytics import YOLO

# Enlace directo a un video de tráfico de prueba válido
URL_VIDEO = "https://github.com/udacity/CarND-Vehicle-Detection/raw/master/project_video.mp4"
NOMBRE_ARCHIVO_LOCAL = "video_entrada.mp4"

print("Descargando video de prueba a Colab...")
opener = urllib.request.build_opener()
opener.addheaders = [('User-agent', 'Mozilla/5.0')]
urllib.request.install_opener(opener)

exito_descarga = False
try:
    urllib.request.urlretrieve(URL_VIDEO, NOMBRE_ARCHIVO_LOCAL)
    if os.path.exists(NOMBRE_ARCHIVO_LOCAL) and os.path.getsize(NOMBRE_ARCHIVO_LOCAL) > 10000:
        print(f"Video descargado correctamente ({os.path.getsize(NOMBRE_ARCHIVO_LOCAL) / (1024*1024):.2f} MB).")
        exito_descarga = True
except Exception as e:
    print(f"No se pudo descargar desde la red: {e}")

# Si la descarga por red falla, generamos un video sintético de contingencia
if not exito_descarga:
    print("Creando un video de prueba interno...")
    ancho, alto, fps_demo = 640, 480, 30
    video_writer = cv2.VideoWriter(NOMBRE_ARCHIVO_LOCAL, cv2.VideoWriter_fourcc(*'mp4v'), fps_demo, (ancho, alto))
    for i in range(120):
        frame_demo = np.zeros((alto, ancho, 3), dtype=np.uint8)
        y_pos = int(50 + i * 3)
        cv2.rectangle(frame_demo, (280, y_pos), (360, y_pos + 60), (0, 0, 255), -1)
        cv2.putText(frame_demo, "CAR", (295, y_pos + 40), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        video_writer.write(frame_demo)
    video_writer.release()
    print("Video sintético creado correctamente.")

# Cargar modelo YOLO
model = YOLO('yolov8n.pt')

print("Procesando detección y cálculo de velocidad...")
cap = cv2.VideoCapture(NOMBRE_ARCHIVO_LOCAL)

fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0 or np.isnan(fps):
    fps = 30

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

DISTANCIA_REAL_METROS = 10.0
LINEA_ENTRADA_Y = int(height * 0.4)
LINEA_SALIDA_Y = int(height * 0.7)

tiempos_entrada = {}
velocidades_registradas = {}

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('resultado_velocidad.mp4', fourcc, fps, (width, height))

frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    tiempo_actual = frame_count / fps

    results = model.track(frame, persist=True, verbose=False)

    cv2.line(frame, (0, LINEA_ENTRADA_Y), (width, LINEA_ENTRADA_Y), (0, 255, 0), 2)
    cv2.line(frame, (0, LINEA_SALIDA_Y), (width, LINEA_SALIDA_Y), (0, 0, 255), 2)

    if results[0].boxes is not None and results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        track_ids = results[0].boxes.id.cpu().numpy().astype(int)
        clases = results[0].boxes.cls.cpu().numpy().astype(int)

        for box, track_id, cls in zip(boxes, track_ids, clases):
            if cls in [2, 3, 5, 7]:
                x1, y1, x2, y2 = map(int, box)
                centro_y = int((y1 + y2) / 2)
                centro_x = int((x1 + x2) / 2)

                if abs(centro_y - LINEA_ENTRADA_Y) < 15:
                    if track_id not in tiempos_entrada:
                        tiempos_entrada[track_id] = tiempo_actual

                if abs(centro_y - LINEA_SALIDA_Y) < 15:
                    if track_id in tiempos_entrada and track_id not in velocidades_registradas:
                        tiempo_transcurrido = tiempo_actual - tiempos_entrada[track_id]
                        if tiempo_transcurrido > 0:
                            velocidad_mps = DISTANCIA_REAL_METROS / tiempo_transcurrido
                            velocidad_kmh = velocidad_mps * 3.6
                            velocidades_registradas[track_id] = velocidad_kmh

                cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 105, 180), 2)

                texto = f"ID: {track_id}"
                if track_id in velocidades_registradas:
                    texto += f" | {velocidades_registradas[track_id]:.1f} km/h"

                cv2.putText(frame, texto, (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
                cv2.circle(frame, (centro_x, centro_y), 4, (0, 0, 255), -1)

    out.write(frame)

cap.release()
out.release()
print("¡Procesamiento finalizado! El video generado es: 'resultado_velocidad.mp4'")

Descargando video de prueba a Colab...
Video descargado correctamente (24.10 MB).
Procesando detección y cálculo de velocidad...
¡Procesamiento finalizado! El video generado es: 'resultado_velocidad.mp4'


In [ ]:
from IPython.display import HTML
from base64 import b64encode

!ffmpeg -i resultado_velocidad.mp4 -vcodec libx264 -f mp4 output_playable.mp4 -y -loglevel quiet

mp4 = open('output_playable.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width=600 controls>
      <source src="{data_url}" type="video/mp4">
</video>
""")

Received > 3 system signals, hard exiting
